# 02.1 — ITCHI con cuatro radios de atribución: ROCLOUD, ROCLOUD medio, W06 (PA21) y Chavas — MSWEP V3.16 a 6 h

Versión del notebook `02_itchi_rocloud_pa21_chavas` con la precipitación y la climatología de MSWEP V3.16. Casos: Cristóbal 2020 (NA) y Odile 2014 (EP); un caso por corrida (`CASE`).

Todo es igual entre modos salvo el radio exterior de atribución:

- `rocloud`: ROCLOUD por cuadrante.
- `rocloud_mean`: media de ROCLOUD, simétrico.
- `pa21`: W06 con metodología PA21, R(2 m/s), simétrico.
- `chavas`: Chavas desde el R34 observado, R(2 m/s), simétrico (`tcwindprofile` si está instalado; si no, E04 propio).

Cambios respecto al 02 (lo demás no cambia: trayectoria, radios, viento de `wind.py`, pipeline, mapas y métricas):

| | 02 | 02.1 |
|---|---|---|
| precipitación | MSWEP V2.8 de 3 h; ventanas de 3 y 6 h | MSWEP V3.16, acumulados de 6 h ya calculados (`D:/MSWEP_V316_6H/mswep_v3_6h_YYYY.nc`), ventana $(T-6\,\mathrm{h},\,T]$ |
| climatología | percentiles diarios V2.8 (`mswep_percentiles_may_nov.nc`) | percentiles de 6 h V3.16 (`mswep_v3_6h_percentiles_may_nov.nc`) |
| rampa $H_P$ | Q90/Q95/Q99 del paquete | igual por defecto (`RAMP = "R3"`); opcional `R4` (Q75/Q90/Q95/Q99) |

La ventana de 3 h desaparece: no hay acumulados de 3 h de V3.16 y no se leen los horarios. La ventana de 6 h del 02 (archivos de 3 h que empiezan en $T-6$ h y $T-3$ h) cubre el mismo intervalo que la de V3.16, así que la comparación 02 (6 h) vs 02.1 aísla el cambio de producto y de climatología.

Un cambio que no es solo de producto: ahora precipitación y percentiles tienen el mismo soporte temporal (6 h contra percentiles de 6 h). En el 02, un acumulado de 3 h se comparaba con percentiles diarios, lo que subestima sistemáticamente $H_P$. Por eso se espera un $H_P$ mayor en el 02.1 sin que la lluvia del evento cambie.

In [ ]:
import os
import re
import sys
import warnings
from dataclasses import replace
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from itchi.aggregation import compute_event_accumulated, compute_event_max
from itchi.config import get_project_root, load_default_config
from itchi.constants import DEFAULT_EPSILON, QUADRANTS
from itchi.mswep import infer_mswep_lat_lon_names, subset_mswep_domain
from itchi.pipeline import compute_itchi_snapshot_from_grid
from itchi.plotting import (
    SIAT_DANGER_LABELS,
    MapStyle,
    configure_map_axis,
    plot_itchi_field,
    plot_itchi_siat_levels,
)
from itchi.precipitation import compute_precipitation_hazard
from itchi.tracks import DEFAULT_R34_COLUMN_MAP

ROOT = get_project_root()
sys.path.insert(0, str(ROOT / "notebooks"))
import itchi.itchi_size_helpers as H  # noqa: E402

## 0. Parámetros

In [ ]:
CFG = load_default_config()
CASES = {
    "CRISTOBAL2020": dict(name="CRISTOBAL", season=2020, basin="NA", atcf="AL032020",
                          t_map="2020-06-04 00:00"),
    "ODILE2014": dict(name="ODILE", season=2014, basin="EP", atcf="EP152014",
                      t_map="2014-09-15 00:00"),
}

In [ ]:
CASE = "ODILE2014"
C = CASES[CASE]
T_MAP = pd.Timestamp(C["t_map"])            # tiempo para los mapas por snapshot

IBTRACS_FILE = ROOT / CFG["paths"]["tracks"] / f"IBTrACS.{C['basin']}.v04r01.nc"
ROCLOUD_FILE = ROOT / "data" / "external" / "Rocloud" / f"ROCLOUD_{C['basin']}.csv"


def local_path(path) -> Path:
    # "D:/..." -> "/mnt/d/..." en WSL/Linux; sin cambios en Windows
    s = str(path).replace("\\", "/")
    if os.name != "nt" and re.match(r"^[A-Za-z]:/", s):
        return Path(f"/mnt/{s[0].lower()}") / s[3:]
    return Path(s)


# MSWEP V3.16, acumulados de 6 h (scripts/mswep_v3_6h_accum.py):
# "precipitation" en mm, time = fin de la ventana, ventana (T-6h, T], solo mayo-noviembre
MSWEP_6H_DIR = local_path("D:/MSWEP_V316_6H")
MSWEP_6H_PATTERN = "mswep_v3_6h_{year}.nc"
PRECIP_VAR = "precipitation"
WINDOW_H = 6
assert MSWEP_6H_DIR.is_dir(), f"no existe {MSWEP_6H_DIR}"
PAD_DEG = 12.0

# Percentiles de 6 h por píxel (MSWEP V3.16, mayo-noviembre)
CLIM_FILE = ROOT / "data" / "processed" / "mswep_v3_6h_percentiles_may_nov.nc"
CLIM_VAR = "p_all"                          # igual que en el 02; "p_wet" para periodos húmedos
CLIM_DOMAIN = dict(lon_min=-140.0, lon_max=-60.0, lat_min=5.0, lat_max=35.0)

# Rampa de H_P: "R3" usa directamente la rampa del paquete (como en el 02)
RAMPS = {
    "R3": {"Q90": 0.0, "Q95": 0.5, "Q99": 1.0},
    "R4": {"Q75": 0.0, "Q90": 0.25, "Q95": 0.5, "Q99": 1.0},
}
RAMP = "R3"

MODES = ("rocloud", "rocloud_mean", "pa21", "chavas")
REFERENCE = "rocloud"
COMMON_TIMES_ONLY = True                    # comparar modos sobre los mismos snapshots

TAG = f"v316_6h_{RAMP}"
FIG_DIR = ROOT / CFG["outputs"]["figures"] / f"itchi_attribution_{CASE.lower()}_{TAG}"
DIAG_DIR = ROOT / CFG["outputs"]["diagnostics"] / f"itchi_attribution_{TAG}"
for d in (FIG_DIR, DIAG_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 1. Trayectoria y radios (IBTrACS, W06 PA21, Chavas, ROCLOUD)

In [ ]:
track = H.load_case_track(IBTRACS_FILE, ROCLOUD_FILE, C["name"], C["season"], C["atcf"])
track = H.add_chavas_radii(track)            # imprime firma/salida de tcwindprofile si existe
print("fuente Chavas:", track["chavas_source"].value_counts().to_dict())

available = pd.DataFrame({
    mode: [H.attribution_radii_km(row, mode)[0] is not None for _, row in track.iterrows()]
    for mode in MODES
}, index=track["time"])
common = available.all(axis=1)
print("snapshots disponibles por modo:", available.sum().to_dict(), "| comunes:", int(common.sum()))
track_eval = track[common.to_numpy()] if COMMON_TIMES_ONLY else track
track[["time", "vmax_kt", "r34_obs_km", "pa21_r2_km", "chavas_r2_km", "rocloud_mean_km"]].round(1)

## 2. MSWEP V3.16: acumulados de 6 h

El snapshot del tiempo sinóptico $T$ es `sel(time=T)` en el archivo anual: la lluvia de las 6 h previas a la posición IBTrACS. Se exige coincidencia exacta del tiempo y se reportan los NaN (ventanas con horas faltantes). La caja se recorta al dominio de los percentiles (5–35°N, 140–60°W).

In [ ]:
bbox = dict(
    lon_min=max(float(track["lon"].min()) - PAD_DEG, CLIM_DOMAIN["lon_min"]),
    lon_max=min(float(track["lon"].max()) + PAD_DEG, CLIM_DOMAIN["lon_max"]),
    lat_min=max(float(track["lat"].min()) - PAD_DEG, CLIM_DOMAIN["lat_min"]),
    lat_max=min(float(track["lat"].max()) + PAD_DEG, CLIM_DOMAIN["lat_max"]),
)


def read_6h_snapshots(times: pd.Series) -> xr.DataArray:
    parts = []
    times = pd.Series(pd.to_datetime(times))
    for year, t_year in times.groupby(times.dt.year):
        path = MSWEP_6H_DIR / MSWEP_6H_PATTERN.format(year=year)
        with xr.open_dataset(path) as ds:
            print(path.name, "|", ds[PRECIP_VAR].attrs.get("accumulation_window"),
                  "|", ds[PRECIP_VAR].attrs.get("units"))
            da = subset_mswep_domain(ds[PRECIP_VAR], **bbox)
            missing_t = t_year[~t_year.isin(pd.DatetimeIndex(da["time"].values))]
            assert missing_t.empty, f"tiempos sin acumulado de 6 h: {list(missing_t)}"
            parts.append(da.sel(time=t_year.values).load())
    return xr.concat(parts, dim="time")


P6 = read_6h_snapshots(track["time"])
LAT_NAME, LON_NAME = infer_mswep_lat_lon_names(P6)
P0 = P6.isel(time=0, drop=True)
LON2D, LAT2D = np.meshgrid(P0[LON_NAME].values, P0[LAT_NAME].values)
PRECIP6 = {pd.Timestamp(t): np.asarray(P6.sel(time=t).values, dtype=float)
           for t in P6["time"].values}

nan_frac = pd.Series({t: np.isnan(v).mean() for t, v in PRECIP6.items()})
print(f"{len(PRECIP6)} snapshots | malla {P0.shape} | "
      f"snapshots con NaN: {int((nan_frac > 0).sum())} (máx. {nan_frac.max():.1%})")

DLAT = abs(float(np.diff(P0[LAT_NAME].values[:2])[0]))
DLON = abs(float(np.diff(P0[LON_NAME].values[:2])[0]))
CELL_AREA_KM2 = (np.deg2rad(DLAT) * H.EARTH_RADIUS_KM) * (
    np.deg2rad(DLON) * H.EARTH_RADIUS_KM * np.cos(np.deg2rad(LAT2D)))

## 3. Climatología de 6 h por píxel (MSWEP V3.16)

Misma lectura que en el 02, ahora sobre `mswep_v3_6h_percentiles_may_nov.nc`. Se verifican la cobertura, el orden de los percentiles y la coherencia de unidades (mm por 6 h en ambos campos).

In [ ]:
clim_ds = xr.open_dataset(CLIM_FILE)
print("variables:", list(clim_ds.data_vars), "| dims:", dict(clim_ds.sizes))
assert CLIM_VAR in clim_ds, f"{CLIM_VAR} no está en {CLIM_FILE.name}"
pw = clim_ds[CLIM_VAR]
qdim = next(d for d in pw.dims if d not in infer_mswep_lat_lon_names(pw))
levels = np.asarray(pw[qdim].values, dtype=float)
scale = 100.0 if levels.max() <= 1.0 else 1.0
clim = xr.Dataset({f"Q{v * scale:g}": pw.sel({qdim: v}, drop=True) for v in levels})
clim_lat, clim_lon = infer_mswep_lat_lon_names(clim)
clim = subset_mswep_domain(clim.rename({clim_lat: LAT_NAME, clim_lon: LON_NAME}), **bbox)
clim = clim.reindex_like(P0, method="nearest", tolerance=1e-3)   # misma malla de 0.1°, sin interpolar

REQUIRED_Q = list(RAMPS[RAMP])
missing_q = [k for k in REQUIRED_Q if k not in clim]
assert not missing_q, f"faltan {missing_q}; disponibles: {list(clim.data_vars)}"
Q = {k: np.asarray(clim[k].values, dtype=float) for k in clim.data_vars}
Q90, Q95, Q99 = Q["Q90"], Q["Q95"], Q["Q99"]
assert Q90.shape == LON2D.shape

ok = np.all([np.isfinite(Q[k]) for k in REQUIRED_Q], axis=0)
ordered = np.all([Q[a] <= Q[b] for a, b in zip(REQUIRED_Q[:-1], REQUIRED_Q[1:])], axis=0)
print(f"cobertura {CLIM_VAR}: {ok.mean():.1%} | orden {' ≤ '.join(REQUIRED_Q)}: {ordered[ok].mean():.2%}")
print("unidades clim:", pw.attrs.get("units"), "| unidades P6:", P6.attrs.get("units"))
pd.Series({k: np.nanmedian(v) for k, v in Q.items()}, name="mediana (mm/6 h)").round(2)

### Rampa de $H_P$

Con `RAMP = "R3"` la precipitación y $Q_{90}, Q_{95}, Q_{99}$ entran directamente a `compute_itchi_snapshot_from_grid`, exactamente como en el 02:

$$
H_P^{R3} = \begin{cases}
0, & P < Q_{90} \\
0.5\,\dfrac{P - Q_{90}}{Q_{95} - Q_{90}}, & Q_{90} \le P < Q_{95} \\[6pt]
0.5 + 0.5\,\dfrac{P - Q_{95}}{Q_{99} - Q_{95}}, & Q_{95} \le P < Q_{99} \\[6pt]
1, & P \ge Q_{99}
\end{cases}
$$

Con `RAMP = "R4"` (nodos $Q_{75} \to 0$, $Q_{90} \to 0.25$, $Q_{95} \to 0.5$, $Q_{99} \to 1$), $H_P$ se calcula aquí y se pasa al pipeline con $(Q_{90}, Q_{95}, Q_{99}) = (0, 0.5, 1)$; con esos valores la rampa del paquete devuelve el mismo $H_P$ que recibe.

In [ ]:
def ramp_hazard(p, qs: dict, nodes: dict):
    # Interpolación lineal por tramos entre nodos (Q_k, h_k); celdas sin percentiles -> 0
    p = np.asarray(p, dtype=float)
    keys = list(nodes)
    qk = [np.asarray(qs[k], dtype=float) for k in keys]
    hk = [float(nodes[k]) for k in keys]
    h = np.zeros(np.broadcast(p, *qk).shape)
    for k in range(len(keys) - 1):
        seg = (p >= qk[k]) & (p < qk[k + 1])
        frac = (p - qk[k]) / np.maximum(qk[k + 1] - qk[k], DEFAULT_EPSILON)
        h = np.where(seg, hk[k] + (hk[k + 1] - hk[k]) * frac, h)
    h = np.where(p >= qk[-1], hk[-1], h)
    valid = np.isfinite(p) & np.all([np.isfinite(q) for q in qk], axis=0)
    return np.clip(np.where(valid, h, 0.0), 0.0, 1.0)


def precip_inputs(time: pd.Timestamp) -> dict:
    # Argumentos de precipitación para el pipeline según la rampa elegida
    if RAMP == "R3":
        return dict(precipitation=PRECIP6[time], q90=Q90, q95=Q95, q99=Q99)
    return dict(precipitation=ramp_hazard(PRECIP6[time], Q, RAMPS[RAMP]), q90=0.0, q95=0.5, q99=1.0)


x = np.linspace(0.0, 1.0, 1001)
assert np.allclose(compute_precipitation_hazard(x, 0.0, 0.5, 1.0), x, atol=1e-9)
print("rampa:", RAMP, RAMPS[RAMP])

## 4. ITCHI por modo de atribución

In [ ]:
FIELDS = ("ITCHI", "H_dir", "H_ind", "H_W", "H_P", "M_direct", "M_indirect")
BOOL_FIELDS = ("M_direct", "M_indirect")


def run_snapshot(row: pd.Series, mode: str):
    radii, n_fixed = H.attribution_radii_km(row, mode)
    if radii is None:
        return None, None, n_fixed
    rmw = float(row["rmw_km"]) if np.isfinite(row["rmw_km"]) else float(row["pa21_rmax_km"])
    fallback = float(row["pa21_r34_km"]) if np.isfinite(row["pa21_r34_km"]) else None
    result = compute_itchi_snapshot_from_grid(
        **precip_inputs(row["time"]),
        lon=LON2D, lat=LAT2D, center_lon=float(row["lon"]), center_lat=float(row["lat"]),
        r34_by_quadrant={q: row[DEFAULT_R34_COLUMN_MAP[q]] for q in QUADRANTS}, r34_unit="nm",
        rocloud_by_quadrant=radii, rocloud_unit="km",
        vmax_kt=float(row["vmax_kt"]), rmw_km=rmw, fallback_direct_radius_km=fallback,
        run_quality_control=True,
    )
    return result, radii, n_fixed


def run_event(mode: str, rows: pd.DataFrame = None):
    rows = track_eval if rows is None else rows
    fields, times, log = [], [], []
    for _, row in rows.iterrows():
        result, radii, n_fixed = run_snapshot(row, mode)
        log.append(dict(time=row["time"], mode=mode, valid=result is not None,
                        quadrants_fixed=n_fixed,
                        attribution_mean_km=np.mean(list(radii.values())) if radii else np.nan))
        if result is not None:
            fields.append({k: np.asarray(result[k], dtype=bool if k in BOOL_FIELDS else np.float32)
                           for k in FIELDS})
            times.append(row["time"])
    dims = ("time", LAT_NAME, LON_NAME)
    coords = {"time": times, LAT_NAME: P0[LAT_NAME].values, LON_NAME: P0[LON_NAME].values}
    ds = xr.Dataset({k: (dims, np.stack([f[k] for f in fields])) for k in FIELDS}, coords=coords)
    ds["ITCHI_max"] = compute_event_max(ds["ITCHI"], dim="time")
    ds["ITCHI_acc"] = compute_event_accumulated(ds["ITCHI"], dim="time")
    ds.attrs.update(case=CASE, atcf=C["atcf"], attribution=mode, window_h=WINDOW_H, ramp=RAMP,
                    precipitation="MSWEP V3.16 6-h accumulation (T-6h, T]",
                    climatology=f"{CLIM_FILE.name}:{CLIM_VAR}",
                    chavas_source=",".join(sorted(set(track["chavas_source"]))))
    return ds, pd.DataFrame(log)


events, logs = {}, []
for mode in MODES:
    events[mode], log = run_event(mode)
    logs.append(log)
logs = pd.concat(logs, ignore_index=True)
logs.groupby("mode")[["valid", "quadrants_fixed"]].sum()

## 5. Mapas del índice

In [ ]:
PC = ccrs.PlateCarree()


def map_axes(n: int, figsize=(17, 4.8)):
    fig, axes = plt.subplots(1, n, figsize=figsize, subplot_kw={"projection": PC}, squeeze=False)
    axes = axes[0]
    for ax in axes:
        ax.coastlines(lw=0.6)
        ax.set_extent([bbox["lon_min"], bbox["lon_max"], bbox["lat_min"], bbox["lat_max"]], crs=PC)
        ax.plot(track["lon"], track["lat"], "k.-", lw=0.8, ms=3, transform=PC)
    return fig, axes


# ITCHI_max del evento por modo
fig, axes = map_axes(len(MODES), figsize=(22, 4.8))
for ax, mode in zip(axes, MODES):
    plot_itchi_field(events[mode]["ITCHI_max"].values, ax, lon=LON2D, lat=LAT2D,
                     crs=PC, colorbar_label="ITCHI_max")
    ax.set_title(f"ITCHI_max — {mode} (V3.16 6 h, {RAMP})", fontsize=10)
fig.savefig(FIG_DIR / "itchi_max_modos.png", dpi=150, bbox_inches="tight")

# Diferencias respecto a ROCLOUD
others = [m for m in MODES if m != REFERENCE]
fig, axes = map_axes(len(others), figsize=(17, 4.8))
for ax, mode in zip(axes, others):
    diff = events[mode]["ITCHI_max"] - events[REFERENCE]["ITCHI_max"]
    m = ax.pcolormesh(LON2D, LAT2D, diff.values, cmap="RdBu_r", vmin=-0.5, vmax=0.5, transform=PC)
    fig.colorbar(m, ax=ax, shrink=0.8)
    ax.set_title(f"ΔITCHI_max ({mode} − {REFERENCE})", fontsize=10)
fig.savefig(FIG_DIR / "itchi_max_diferencias.png", dpi=150, bbox_inches="tight")

In [ ]:
SNAP_TIME = pd.Timestamp(C["t_map"])        # posición a mapear (cámbiala libremente)
SNAP_MODES = MODES
SNAP_FIELDS = ("H_dir", "H_ind", "ITCHI", "SIAT")
SNAP_HALF_WIDTH_DEG = 14.0
SNAP_STYLE = MapStyle(
    lon_tick_spacing=8.0, lat_tick_spacing=8.0, grid_spacing=4.0,
    grid_color="w", grid_alpha=0.6, label_size=10.0,
)
CBAR = dict(height=0.70, width=0.035, pad=0.03, labelsize=9)   # fracciones del eje del mapa


def snapshot_style(row: pd.Series, base: MapStyle = SNAP_STYLE,
                   half_width: float = SNAP_HALF_WIDTH_DEG) -> MapStyle:
    # Extensión centrada en el ciclón, alineada a múltiplos del espaciado de ticks
    step = base.lon_tick_spacing
    lon0 = np.floor((row["lon"] - half_width) / step) * step
    lon1 = np.ceil((row["lon"] + half_width) / step) * step
    lat0 = np.floor((row["lat"] - 0.75 * half_width) / step) * step
    lat1 = np.ceil((row["lat"] + 0.75 * half_width) / step) * step
    return replace(base, lon_extent=(lon0, lon1), lat_extent=(lat0, lat1))


def overlay_radii(ax, row: pd.Series, attribution: dict | None) -> None:
    ax.plot(track["lon"], track["lat"], "k-", lw=0.8, transform=PC, zorder=5)
    x, y = H.quadrant_outline(row["lat"], row["lon"], H.direct_radii_km(row))
    ax.plot(x, y, "k-", lw=1.4, transform=ccrs.Geodetic(), zorder=6, label="radio directo")
    if attribution is not None:
        x, y = H.quadrant_outline(row["lat"], row["lon"], attribution)
        ax.plot(x, y, "m-", lw=1.6, transform=ccrs.Geodetic(), zorder=6, label="atribución")
    ax.plot(row["lon"], row["lat"], "k*", ms=10, transform=PC, zorder=7)


def add_small_colorbar(ax, mappable, label: str, ticks=None, ticklabels=None, **opts):
    # Colorbar en un eje insertado: alto = fracción del alto del mapa, centrado
    o = {**CBAR, **opts}
    cax = ax.inset_axes([1.0 + o["pad"], (1.0 - o["height"]) / 2, o["width"], o["height"]])
    cbar = ax.figure.colorbar(mappable, cax=cax, ticks=ticks)
    cbar.set_label(label, fontsize=o["labelsize"])
    cbar.ax.tick_params(labelsize=o["labelsize"] - 1)
    if ticklabels is not None:
        cbar.set_ticklabels(ticklabels)
    return cbar


def draw_field(ax, field: str, result: dict, row: pd.Series, radii: dict | None,
               cbar_opts: dict | None = None):
    cbar_opts = cbar_opts or {}
    precip = PRECIP6[row["time"]]
    if field == "P":
        wet = np.where(precip >= 0.1, precip, np.nan)
        mesh = ax.pcolormesh(LON2D, LAT2D, wet, cmap="YlGnBu", shading="auto",
                             transform=PC, zorder=1)
        add_small_colorbar(ax, mesh, f"P ({WINDOW_H} h, mm)", **cbar_opts)
        ax.contour(LON2D, LAT2D, precip - Q95, levels=[0.0], colors="r",
                   linewidths=0.7, transform=PC, zorder=5)            # P = Q95 de 6 h
    elif field == "SIAT":
        mesh = plot_itchi_siat_levels(result["ITCHI"], ax, lon=LON2D, lat=LAT2D, crs=PC,
                                      add_colorbar=False, zorder=1)
        add_small_colorbar(ax, mesh, "nivel ITCHI", ticks=np.arange(0.5, 6.5, 1.0),
                           ticklabels=[lab.split(" ", 1)[0] for lab in SIAT_DANGER_LABELS],
                           **cbar_opts)
    else:
        values = np.asarray(result[field], dtype=float)
        mesh = plot_itchi_field(np.where(values > 0, values, np.nan), ax, lon=LON2D,
                                lat=LAT2D, crs=PC, add_colorbar=False, zorder=1)
        add_small_colorbar(ax, mesh, field, **cbar_opts)
    overlay_radii(ax, row, radii)


def plot_position_snapshot(time: pd.Timestamp, modes=SNAP_MODES, fields=SNAP_FIELDS):
    # Rejilla (modos x campos) con colorbars pequeñas
    row = track.loc[track["time"] == time].iloc[0]
    style = snapshot_style(row)
    fig, axes = plt.subplots(len(modes), len(fields),
                             figsize=(5.0 * len(fields), 4.2 * len(modes)),
                             subplot_kw={"projection": PC}, squeeze=False)
    fig.subplots_adjust(wspace=0.45, hspace=0.25)
    for i, mode in enumerate(modes):
        result, radii, _ = run_snapshot(row, mode)
        for j, field in enumerate(fields):
            ax = configure_map_axis(axes[i, j], crs=PC, style=style)
            if result is None:
                ax.set_title(f"{mode}: sin radio de atribución válido", fontsize=9)
                continue
            draw_field(ax, field, result, row, radii)
            ax.set_title(f"{field} — {mode}", fontsize=11)
    axes[0, 0].legend(fontsize=8, loc="lower left")
    fig.suptitle(f"{CASE} {time:%Y-%m-%d %H} UTC | Vmax {row['vmax_kt']:.0f} kt | "
                 f"MSWEP V3.16 {WINDOW_H} h | {RAMP}", fontsize=13)
    return fig


def plot_position_field(time: pd.Timestamp, mode: str, field: str,
                        half_width: float = SNAP_HALF_WIDTH_DEG, figsize=(9, 7),
                        style: MapStyle | None = None, cbar_opts=None):
    # Un solo mapa a detalle: una posición, un modo de atribución, un campo
    row = track.loc[track["time"] == time].iloc[0]
    result, radii, _ = run_snapshot(row, mode)
    if result is None:
        warnings.warn(f"{mode}: sin radio de atribución válido en {time}; se omite el mapa")
        return None
    base = style or replace(SNAP_STYLE, label_size=12.0, tick_size=7.0)
    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": PC})
    configure_map_axis(ax, crs=PC, style=snapshot_style(row, base, half_width))
    draw_field(ax, field, result, row, radii, cbar_opts={"labelsize": 11, **(cbar_opts or {})})
    ax.legend(fontsize=9, loc="lower left")
    ax.set_title(f"{field} — atribución {mode}\n{CASE} {time:%Y-%m-%d %H} UTC | "
                 f"Vmax {row['vmax_kt']:.0f} kt | MSWEP V3.16 {WINDOW_H} h | {RAMP}", fontsize=12)
    return fig

In [ ]:
# Rejilla completa con barras pequeñas
fig = plot_position_snapshot(SNAP_TIME)
fig.savefig(FIG_DIR / f"snapshot_{SNAP_TIME:%Y%m%d%H}.png", dpi=150, bbox_inches="tight")

In [ ]:
# Un mapa a detalle por campo y modo
fig = plot_position_field(SNAP_TIME, mode="rocloud", field="H_ind", half_width=10)
fig = plot_position_field(SNAP_TIME, mode="pa21", field="P", cbar_opts={"height": 0.5})

# Todos los campos y modos de una posición, cada uno en su propia figura
for mode in MODES:
    for field in SNAP_FIELDS:
        fig = plot_position_field(SNAP_TIME, mode, field)
        if fig is None:
            continue
        fig.savefig(FIG_DIR / f"{SNAP_TIME:%Y%m%d%H}_{mode}_{field}.png",
                    dpi=150, bbox_inches="tight")
        plt.close(fig)

## 6. Métricas del índice por modo

Áreas en km² (celdas ponderadas por cos φ). Las métricas de diferencia se calculan sobre la unión de las huellas (ITCHI_max > 0 en el modo o en la referencia).

In [ ]:
def event_metrics(ev: xr.Dataset, ref: xr.Dataset) -> dict:
    a = CELL_AREA_KM2
    x = np.nan_to_num(ev["ITCHI_max"].values)
    y = np.nan_to_num(ref["ITCHI_max"].values)
    direct = ev["M_direct"].any("time").values
    indirect = ev["M_indirect"].any("time").values & ~direct
    h_ind = np.nan_to_num(ev["H_ind"].max("time").values)
    union = (x > 0) | (y > 0)
    d, w = (x - y)[union], a[union]
    both, either = (x >= 0.5) & (y >= 0.5), (x >= 0.5) | (y >= 0.5)
    footprint = x > 0
    return {
        "snapshots": int(ev.sizes["time"]),
        "area_directa_km2": a[direct].sum(),
        "area_indirecta_km2": a[indirect].sum(),
        "area_ITCHI>=0.25_km2": a[x >= 0.25].sum(),
        "area_ITCHI>=0.50_km2": a[x >= 0.50].sum(),
        "area_ITCHI>=0.75_km2": a[x >= 0.75].sum(),
        "ITCHI_integrado_km2": (x * a).sum(),
        "ITCHI_max_medio_en_huella": (x * a)[footprint].sum() / a[footprint].sum(),
        "H_ind_medio_en_indirecta": (h_ind * a)[indirect].sum() / max(a[indirect].sum(), 1e-9),
        "dITCHI_medio_vs_ref": np.average(d, weights=w) if w.sum() else np.nan,
        "RMSD_vs_ref": np.sqrt(np.average(d**2, weights=w)) if w.sum() else np.nan,
        "frac_area_|d|>0.1": w[np.abs(d) > 0.1].sum() / w.sum() if w.sum() else np.nan,
        "Jaccard_ITCHI>=0.5": a[both].sum() / a[either].sum() if a[either].sum() else np.nan,
    }


metrics = pd.DataFrame([{"case": CASE, "product": TAG, "mode": mode,
                         **event_metrics(ev, events[REFERENCE])}
                        for mode, ev in events.items()]).set_index(["case", "product", "mode"])
metrics.to_csv(DIAG_DIR / f"metricas_evento_{CASE.lower()}.csv")
metrics.round(3)

In [ ]:
# Métricas por snapshot: área indirecta e ITCHI integrado
per_time = []
for mode, ev in events.items():
    a = CELL_AREA_KM2
    for t in ev["time"].values:
        snap = ev.sel(time=t)
        indirect = snap["M_indirect"].values & ~snap["M_direct"].values
        per_time.append(dict(case=CASE, mode=mode, time=pd.Timestamp(t),
                             area_indirecta_km2=a[indirect].sum(),
                             ITCHI_integrado_km2=(np.nan_to_num(snap["ITCHI"].values) * a).sum(),
                             H_ind_max=float(np.nanmax(snap["H_ind"].values))))
per_time = pd.DataFrame(per_time).merge(
    logs[["time", "mode", "attribution_mean_km"]], on=["time", "mode"])
per_time.to_csv(DIAG_DIR / f"metricas_snapshot_{CASE.lower()}.csv", index=False)

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
for mode in MODES:
    d = per_time[per_time["mode"] == mode]
    axes[0].plot(d["time"], d["attribution_mean_km"], label=mode)
    axes[1].plot(d["time"], d["area_indirecta_km2"] / 1e3, label=mode)
    axes[2].plot(d["time"], d["ITCHI_integrado_km2"] / 1e3, label=mode)
axes[0].set_ylabel("radio de atribución\nmedio (km)")
axes[1].set_ylabel("área indirecta\n(10³ km²)")
axes[2].set_ylabel("ITCHI integrado\n(10³ km²)")
axes[0].legend()
fig.autofmt_xdate()
fig.savefig(FIG_DIR / "metricas_snapshot.png", dpi=150, bbox_inches="tight")

## 7. Tabla conjunta (tras correr ambos casos) y comparación con el 02

La primera tabla junta los casos corridos con esta configuración. La segunda pone lado a lado el 02 (V2.8; ventana de 6 h si se corrió) y el 02.1 (V3.16), por caso y modo.

In [ ]:
paths = sorted(DIAG_DIR.glob("metricas_evento_*.csv"))
pd.concat([pd.read_csv(p) for p in paths]).round(3) if paths else None

In [ ]:
# 02 (V2.8) frente a 02.1 (V3.16): mismas métricas por caso y modo
old_dir = ROOT / CFG["outputs"]["diagnostics"] / "itchi_attribution"
old_path = old_dir / f"metricas_evento_{CASE.lower()}.csv"
if old_path.exists():
    old = pd.read_csv(old_path)
    if "window_h" in old:
        old = old[old["window_h"] == 6]
    old = old.assign(product="v280").set_index(["case", "mode"])
    new = metrics.reset_index().set_index(["case", "mode"])
    cols = ["area_indirecta_km2", "area_ITCHI>=0.50_km2", "ITCHI_integrado_km2",
            "ITCHI_max_medio_en_huella", "H_ind_medio_en_indirecta"]
    display(pd.concat({"02 (V2.8)": old[cols], "02.1 (V3.16)": new[cols]}, axis=1).round(3))
else:
    print("no hay métricas del 02 para", CASE)